In [2]:
# ── Cell 1: install ──────────────────────────────────────────────
!pip install -q --upgrade kaggle tensorflowjs scikit-learn
print("Done")

Done


In [9]:
# ── Cell 2: download the Kaggle palm dataset ─────────────────────
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_65e85231a11a8f5bebe195553bb3c833"
!kaggle datasets download -d husamalzain/palm-disease-dataset -p data --unzip

TARGET = "data/palm-disease-dataset (all jpg)"
import glob
for d in sorted(os.listdir(TARGET)):
    p = os.path.join(TARGET, d)
    if os.path.isdir(p):
        print(d, "→", len(glob.glob(os.path.join(p, "*"))), "images")

Dataset URL: https://www.kaggle.com/datasets/husamalzain/palm-disease-dataset
License(s): MIT
100% 1.52G/1.52G [00:16<00:00, 99.2MB/s]

Dryness → 70 images
Fungal disease → 59 images
Magnesium Deficiency → 59 images
Scale insect → 142 images


In [10]:
# ── Cell 3: add HEALTHY class from the second Kaggle dataset ─────
!kaggle datasets download -d hadjerhamaidi/date-palm-data -p healthy_src --unzip

healthy_dir = None
for root, dirs, fs in os.walk("healthy_src"):
    if os.path.basename(root).lower() in ("healthy", "healthy leaves", "healthy_leaves"):
        if any(f.lower().endswith((".jpg",".jpeg",".png")) for f in fs):
            healthy_dir = root; break
print("Healthy images found in:", healthy_dir)

dest = os.path.join(TARGET, "Healthy")
os.makedirs(dest, exist_ok=True)
imgs = [f for f in glob.glob(os.path.join(healthy_dir, "*"))
        if f.lower().endswith((".jpg",".jpeg",".png"))]
import random; random.seed(123)
if len(imgs) > 150: imgs = random.sample(imgs, 150)
for i, src in enumerate(imgs):
    shutil.copy(src, os.path.join(dest, f"healthy_{i}.jpg"))
print(f"Added {len(imgs)} healthy images")

Dataset URL: https://www.kaggle.com/datasets/hadjerhamaidi/date-palm-data
License(s): CC-BY-NC-SA-4.0
100% 42.7M/42.7M [00:00<00:00, 54.9MB/s]

Healthy images found in: healthy_src/date palm data/Date Palm data/healthy
Added 150 healthy images


In [7]:
import os, glob, shutil

my_fungal = "/content/drive/MyDrive/fungal"   # <-- your folder of fungal images
TARGET = "data/palm-disease-dataset (all jpg)"
dst = os.path.join(TARGET, "Fungal disease")  # the existing class folder

print("Source exists?", os.path.isdir(my_fungal))
n = 0
for img in glob.glob(os.path.join(my_fungal, "*")):
    if img.lower().endswith((".jpg",".jpeg",".png",".bmp",".webp")):
        shutil.copy(img, os.path.join(dst, "added_" + os.path.basename(img)))
        n += 1
print(f"Added {n} fungal images")

# confirm new counts
for d in sorted(os.listdir(TARGET)):
    p = os.path.join(TARGET, d)
    if os.path.isdir(p):
        print(" ", d, "→", len(glob.glob(os.path.join(p, "*"))), "images")

Source exists? True
Added 34 fungal images
  Dryness → 70 images
  Fungal disease → 59 images
  Magnesium Deficiency → 59 images
  Scale insect → 142 images


In [11]:
import os, glob
TARGET = "data/palm-disease-dataset (all jpg)"
print("All folders:", os.listdir(TARGET))
h = os.path.join(TARGET, "Healthy")
print("Healthy exists?", os.path.isdir(h))
if os.path.isdir(h):
    print("Healthy images:", len(glob.glob(os.path.join(h, "*"))))

All folders: ['Fungal disease', 'Healthy', 'Dryness', 'Magnesium Deficiency', 'Scale insect']
Healthy exists? True
Healthy images: 150


In [12]:
# ── Cell 3: merge YOUR fungal images from Drive ──────────────────
# In Drive: right-click your palms folder → Organize → Add shortcut → My Drive.
from google.colab import drive
drive.mount('/content/drive')

import shutil
my_palms = "/content/drive/MyDrive/fungal"   # <-- change to YOUR folder path/name

# match each Drive subfolder to the existing class by keyword (no silent skips)
def match_class(name):
    n = name.lower()
    if "fungal" in n:      return "Fungal disease"
    if "dry" in n:         return "Dryness"
    if "magnes" in n:      return "Magnesium Deficiency"
    if "scale" in n:       return "Scale insect"
    if "healthy" in n:     return "Healthy"
    return None

if os.path.isdir(my_palms):
    for cls in os.listdir(my_palms):
        src = os.path.join(my_palms, cls)
        if not os.path.isdir(src): continue
        target_cls = match_class(cls)
        if target_cls is None:
            print(f"⚠ skipped unmatched folder: {cls}")
            continue
        dst = os.path.join(TARGET, target_cls)
        n = 0
        for img in glob.glob(os.path.join(src, "*")):
            if img.lower().endswith((".jpg",".jpeg",".png",".bmp",".webp")):
                shutil.copy(img, os.path.join(dst, "added_" + os.path.basename(img)))
                n += 1
        print(f"  added {n} images to '{target_cls}'")
else:
    print("⚠ Drive folder not found — check the path:", my_palms)

# cap Healthy so it doesn't dominate, then show final counts
healthy = os.path.join(TARGET, "Healthy")
h = glob.glob(os.path.join(healthy, "*.jpg"))
import random; random.seed(123)
CAP = 150
if len(h) > CAP:
    for f in random.sample(h, len(h) - CAP): os.remove(f)

print("\nFinal counts:")
for d in sorted(os.listdir(TARGET)):
    p = os.path.join(TARGET, d)
    if os.path.isdir(p):
        print(" ", d, "→", len(glob.glob(os.path.join(p, "*"))), "images")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Final counts:
  Dryness → 70 images
  Fungal disease → 59 images
  Healthy → 150 images
  Magnesium Deficiency → 59 images
  Scale insect → 142 images


In [16]:
from PIL import Image
import glob, os

TARGET = "data/palm-disease-dataset (all jpg)"
converted, removed = 0, 0
for cls in os.listdir(TARGET):
    cdir = os.path.join(TARGET, cls)
    if not os.path.isdir(cdir): continue
    for f in glob.glob(os.path.join(cdir, "*")):
        try:
            with Image.open(f) as im:
                rgb = im.convert("RGB")        # strips alpha, fixes CMYK, etc.
            os.remove(f)                        # remove original (any format)
            rgb.save(os.path.splitext(f)[0] + ".jpg", "JPEG", quality=90)
            converted += 1
        except Exception:
            os.remove(f); removed += 1          # truly unreadable -> drop

print(f"Converted {converted} images to clean JPEG, removed {removed} unreadable.")
for d in sorted(os.listdir(TARGET)):
    p = os.path.join(TARGET, d)
    if os.path.isdir(p):
        print(" ", d, "→", len(glob.glob(os.path.join(p, '*'))), "images")

Converted 480 images to clean JPEG, removed 0 unreadable.
  Dryness → 70 images
  Fungal disease → 59 images
  Healthy → 150 images
  Magnesium Deficiency → 59 images
  Scale insect → 142 images


In [17]:
# ── Cell 5: build datasets + train (5-class, early stopping) ─────
import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np

DATA_ROOT = TARGET
IMG_SIZE, BATCH, SEED = 224, 32, 123
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

classes = sorted([d for d in os.listdir(DATA_ROOT)
                  if os.path.isdir(os.path.join(DATA_ROOT, d))])
train_paths, train_lbls, val_paths, val_lbls = [], [], [], []
for ci, c in enumerate(classes):
    fs = [f for f in glob.glob(os.path.join(DATA_ROOT, c, "*"))
          if f.lower().endswith((".jpg",".jpeg",".png",".bmp",".webp"))]
    random.shuffle(fs)
    k = max(1, int(len(fs)*0.2))
    val_paths += fs[:k];   val_lbls += [ci]*k
    train_paths += fs[k:]; train_lbls += [ci]*(len(fs)-k)

CLASS_NAMES = classes; NUM_CLASSES = len(classes)
print("Classes:", CLASS_NAMES)
print("Train per class:", np.bincount(train_lbls, minlength=NUM_CLASSES).tolist())
print("Val per class:  ", np.bincount(val_lbls,   minlength=NUM_CLASSES).tolist())

def load(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    return tf.image.resize(img, (IMG_SIZE, IMG_SIZE)), label

def make_ds(paths, lbls, training):
    ds = tf.data.Dataset.from_tensor_slices((paths, lbls))
    if training: ds = ds.shuffle(len(paths), seed=SEED)
    return ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)

augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"), layers.RandomRotation(0.12),
    layers.RandomZoom(0.12), layers.RandomContrast(0.12)])

train_ds = make_ds(train_paths, train_lbls, True)
train_ds = train_ds.map(lambda x,y:(augment(x,training=True),y),
                        num_parallel_calls=tf.data.AUTOTUNE)
val_ds = make_ds(val_paths, val_lbls, False)

counts = np.bincount(train_lbls, minlength=NUM_CLASSES)
class_weight = {i: float(counts.sum()/(NUM_CLASSES*max(counts[i],1))) for i in range(NUM_CLASSES)}

inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3))
x = layers.Rescaling(1./127.5, offset=-1)(inputs)
base = tf.keras.applications.MobileNetV2(include_top=False, weights="imagenet",
                                         input_shape=(IMG_SIZE,IMG_SIZE,3))
base.trainable = False
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = Model(inputs, outputs)
model.compile(tf.keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy", metrics=["accuracy"])

stop = tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=6,
                                        restore_best_weights=True)

print("\n--- Phase 1: head ---")
model.fit(train_ds, validation_data=val_ds, epochs=25,
          class_weight=class_weight, callbacks=[stop])

print("\n--- Phase 2: fine-tune ---")
base.trainable = True
for l in base.layers[:-50]: l.trainable = False
model.compile(tf.keras.optimizers.Adam(1e-5), "sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(train_ds, validation_data=val_ds, epochs=15,
          class_weight=class_weight, callbacks=[stop])

Classes: ['Dryness', 'Fungal disease', 'Healthy', 'Magnesium Deficiency', 'Scale insect']
Train per class: [56, 48, 120, 48, 114]
Val per class:   [14, 11, 30, 11, 28]

--- Phase 1: head ---
Epoch 1/25
13/13 ━━━━━━━━━━━━━━━━━━━━ 58s 4s/step - accuracy: 0.4326 - loss: 1.5961 - val_accuracy: 0.5957 - val_loss: 1.0458
Epoch 2/25
13/13 ━━━━━━━━━━━━━━━━━━━━ 47s 4s/step - accuracy: 0.5881 - loss: 1.1990 - val_accuracy: 0.6277 - val_loss: 0.9240
Epoch 3/25
13/13 ━━━━━━━━━━━━━━━━━━━━ 47s 4s/step - accuracy: 0.6347 - loss: 1.1157 - val_accuracy: 0.6383 - val_loss: 0.8595
Epoch 4/25
13/13 ━━━━━━━━━━━━━━━━━━━━ 47s 4s/step - accuracy: 0.6399 - loss: 1.0223 - val_accuracy: 0.6596 - val_loss: 0.8220
Epoch 5/25
13/13 ━━━━━━━━━━━━━━━━━━━━ 46s 3s/step - accuracy: 0.6736 - loss: 1.0162 - val_accuracy: 0.6809 - val_loss: 0.8105
Epoch 6/25
13/13 ━━━━━━━━━━━━━━━━━━━━ 46s 4s/step - accuracy: 0.7073 - loss: 0.8723 - val_accuracy: 0.6702 - val_loss: 0.8162
Epoch 7/25
13/13 ━━━━━━━━━━━━━━━━━━━━ 48s 4s/step - a

In [18]:
# ── Cell 6: evaluate — 5-class AND the binary headline number ────
from sklearn.metrics import classification_report, confusion_matrix

y_true = np.concatenate([y.numpy() for _, y in val_ds])
y_pred = model.predict(val_ds).argmax(axis=1)

print("===== 5-class (detail behind the hint) =====")
print(f"Accuracy: {(y_pred==y_true).mean()*100:.1f}%  on {len(y_true)} held-out images")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3, zero_division=0))

hi = CLASS_NAMES.index("Healthy")
yt_bin = (y_true != hi).astype(int)
yp_bin = (y_pred != hi).astype(int)
print("\n===== Healthy vs Diseased (THE number you report) =====")
print(f"Accuracy: {(yt_bin==yp_bin).mean()*100:.1f}%  on {len(yt_bin)} held-out images")
print(classification_report(yt_bin, yp_bin, target_names=["Healthy","Diseased"], digits=3, zero_division=0))
print("Confusion matrix [rows=actual, cols=pred] (Healthy, Diseased):")
print(confusion_matrix(yt_bin, yp_bin))

3/3 ━━━━━━━━━━━━━━━━━━━━ 11s 3s/step
===== 5-class (detail behind the hint) =====
Accuracy: 70.2%  on 94 held-out images
                      precision    recall  f1-score   support

             Dryness      0.400     0.714     0.513        14
      Fungal disease      0.636     0.636     0.636        11
             Healthy      1.000     1.000     1.000        30
Magnesium Deficiency      0.500     0.364     0.421        11
        Scale insect      0.750     0.536     0.625        28

            accuracy                          0.702        94
           macro avg      0.657     0.650     0.639        94
        weighted avg      0.735     0.702     0.705        94


===== Healthy vs Diseased (THE number you report) =====
Accuracy: 100.0%  on 94 held-out images
              precision    recall  f1-score   support

     Healthy      1.000     1.000     1.000        30
    Diseased      1.000     1.000     1.000        64

    accuracy                          1.000        94
   

In [19]:
# ── Cell 7: prediction — confident call + "most likely" hint ─────
def predict_palm(image_path):
    img = tf.image.resize(
        tf.io.decode_image(tf.io.read_file(image_path), channels=3,
                           expand_animations=False), (IMG_SIZE, IMG_SIZE))
    probs = model.predict(img[None, ...], verbose=False)[0]
    top_i = int(probs.argmax())
    if CLASS_NAMES[top_i] == "Healthy":
        return {"status": "Healthy", "detail": None, "confidence": float(probs[top_i])}
    diseases = [(CLASS_NAMES[i], float(p)) for i, p in enumerate(probs) if CLASS_NAMES[i] != "Healthy"]
    diseases.sort(key=lambda x: -x[1])
    likely, conf = diseases[0]
    return {"status": "Diseased", "detail": f"most likely {likely}", "confidence": conf}

# test on a FRESH image the model never trained on:
from google.colab import files
up = files.upload()
for name in up:
    print(name, "→", predict_palm(name))

KeyboardInterrupt: 

In [20]:
# ── Cell 8: export for the app + download ────────────────────────
import tensorflowjs as tfjs, json, zipfile
OUT = "models/palm"; shutil.rmtree(OUT, ignore_errors=True); os.makedirs(OUT, exist_ok=True)
tfjs.converters.save_keras_model(model, OUT)
json.dump(CLASS_NAMES, open(os.path.join(OUT, "labels.json"), "w"), ensure_ascii=False)
model.save("palm_model.keras")
with zipfile.ZipFile("palm_model.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(os.path.join(OUT, "*")):
        z.write(f, os.path.join("palm", os.path.basename(f)))
    z.write("palm_model.keras")
files.download("palm_model.zip")
print("Exported — labels:", CLASS_NAMES)

KeyboardInterrupt: 